# MultiIndex Tasks — Exercises and Solutions (Task 21)

This task set follows the MultiIndex session and asks you to put hierarchical indexing to work on three genuinely real problems. MultiIndexes let a single DataFrame carry several keys on the same axis — branch and year, country and province, country and city — and the questions here exercise both the *mechanics* of the machinery and its *practical* value.

Q-1 is pure index surgery: a fabricated branch × year DataFrame whose columns also carry a two-level front (city x metric). Your job is to **swap the level-0 of the index with the level-0 of the columns** — effectively turning `(cse,ece) rows × (delhi,mumbai) columns` into `(delhi,mumbai) rows × (cse,ece) columns` — and then sort the new row index. It is a clean drill of `swaplevel(axis=1)`, column-level reassembly, and `sort_index(level=..., ascending=...)`.

Q-2 through Q-4 attack the COVID-19 global time series. Two Google-Sheets-hosted CSVs give per-day confirmed cases and deaths per province and country, each in a wide format with one column per date. Q-2 asks you to melt both, merge them, and rebuild a tidy MultiIndex table keyed by **Country (level-0) and Province/State (level-1)**. Q-3 then resurrects the natural analytical question — *which country has the highest death percentage out of confirmed cases?* — demanding a groupwise rate after the reshaping. Q-4 zooms into India and introduces one of the most useful Pandas features for time series: the **`shift`** function, which lets you compute *new cases today = cases today − cases yesterday* in a single expression.

Finally Q-5 imports the world's most polluted cities dataset and builds a MultiIndex on `Country` and `City/Town`. You answer questions with `.loc` lookups and group operations (India's worst city by PM10, China's cleanest, Poland's share of pollution as a pie chart, Israel and Qatar as bars), then collapse the whole structure back to a Series with `stack`/`unstack`. Each question is an opportunity to glue names to resolves — good luck.

### ✓ How to use this notebook

Every question ends with a `# code here` cell. Type your answer there, run it, and compare against the values mentioned in the question text.

In [2]:
import pandas as pd

# Task 1 — Swap Index and Column Levels

### Q-1 — Restructure the branch DataFrame

**Goal:** A DataFrame indexed by `(branch, year)` with two-level columns `(city, metric)`. Swap the axis levels so the **index** becomes `(delhi, mumbai)` and the **columns** become `(cse, ece)`, then sort the row index by its first level.

###`Q-1:` You are given a Multi index DataFrame. You task is to swap level-0 index with level-0 column.
* Change Branch -> ('cse', 'ece') as columns at level-0 and ('delhi'	'mumbai') as level-0 Index
* And Sort on row index level-0

In [3]:
# Given Code Snippets
index_val = [('cse',2019),('cse',2020),('cse',2021),('cse',2022),('ece',2019),('ece',2020),('ece',2021),('ece',2022)]
multiindex = pd.MultiIndex.from_tuples(index_val)
df = pd.DataFrame(
    [
        [1,2,0,0],
        [3,4,0,0],
        [5,6,0,0],
        [7,8,0,0],
        [9,10,0,0],
        [11,12,0,0],
        [13,14,0,0],
        [15,16,0,0],
    ],
    index = multiindex,
    columns = pd.MultiIndex.from_product([['delhi','mumbai'],['avg_package','students']])
)

df

               delhi               mumbai         
         avg_package students avg_package students
cse 2019           1        2           0        0
    2020           3        4           0        0
    2021           5        6           0        0
    2022           7        8           0        0
ece 2019           9       10           0        0
    2020          11       12           0        0
    2021          13       14           0        0
    2022          15       16           0        0

## 1.1 Approach — swaplevel then rebuild the columns

The given frame has 8 rows labelled `(cse 2019) ... (ece 2022)` and column names formed by `MultiIndex.from_product([['delhi','mumbai'],['avg_package','students']])`. Levels live on the columns axis: the *outer* level is the city, the *inner* is the metric.

The natural solution is **`swaplevel(axis=1)`**: this swaps the two column levels so every column name becomes `(metric, city)` — `('avg_package','delhi')`, `('students','mumbai')`, and so on. Crucially, the data itself does not move; only the labels are re-arranged.

But the target states the *index* should be the cities and the *columns* the branches — a full transposition plus level juggling. The cleanest path mirrors what was demoed in Session 21: after `swaplevel`, **sort the columns** so the metric becomes the outer level, then **transpose** the frame (`df.T`). Transposing flips rows and columns, promoting the two-level rows (`branch, year`) into a two-level column header while demoting the `(metric, city)` columns into the row index. Finally **`sort_index(level=0, ascending=[False])`** puts the new row index (the cities) in the requested order.

Checklist for your solution:
1. `swaplevel(axis=1)` — swap the two column levels.
2. `sort_index(axis=1, level=0)` — metric outer, city inner (adjustable via `ascending`).
3. `.T` — transpose rows and columns.
4. `sort_index(level=0)` — order the new row index by level-0.
5. Verify the resulting header reads `cse` and `ece` at level-0 of the columns and `delhi`/`mumbai` label the rows.

In [ ]:
# code here

# Task 2 — COVID Data as a MultiIndex

### Q-2 — Country + Province/State index

**Goal:** Using the two COVID CSVs (confirmed cases and deaths), build `datedata` — a DataFrame indexed by **Country (level-0)** and **Province/State (level-1)** with columns `date`, `No of cases`, and `No of Deaths`.

###`Q-2:` Covid Cases Data Set Problem
 Make a DataFrame Using both Covid Dataset. With Country name as Level-0 and Provinance/State as Level-1 Index and date, No of cases and No of Deaths as Columns.

## 2.1 Approach — from wide tables to a tidy MultiIndex frame

The two CSVs are already loaded (`confirm` for cases, `deaths` for fatalities) in *wide* form — one row per geolocation, one **column per date**. To build the requested tidy table you must reverse that shape.

1. **Melt both frames** with `pd.melt(...)`, passing `id_vars=['Province/State', 'Country/Region', 'Lat', 'Long']`, `var_name='date'`, and values renamed `No of cases` (for `confirm`) and `No of Deaths` (for `deaths`). Every daily column collapses into rows of `(place, date, value)`.
2. **Merge** the two melted frames on the five key columns — `['Province/State', 'Country/Region', 'Lat', 'Long', 'date']` — so each `(place, date)` row carries both cases and deaths. A one-to-one merge on these five keys joins the rows correctly, merging as much as possible ('outer') so no location is dropped.
3. **Set the hierarchical index** with `set_index(['Country/Region', 'Province/State'])` (you may drop the coordinates first), producing exactly the requested `level-0 = Country, level-1 = Province/State`.

Before writing code, inspect both raw frames (`head`, `shape`, `info`) to confirm the column names — spelling of `Province/State` versus `Province_State` must match across both merges or the join will multiply rows. This is a classic real-data gotcha; plan for it.

In [4]:
confirm = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vTmqT3kxd0i0RUFiEnwA1Hboiunv28MeNTatZsIEqlPPB7mHrl0ttJL7utZ23_1s5FW8ZjODmB8jHIi/pub?gid=2142019845&single=true&output=csv')

deaths = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQn4nLCKIVZMw4e89YeRqgKaSAAwRFaZ6ILMW_FUejZ2tkv3Np1f9gD4QOl3ASHeLzWjOjfmLDKcCOF/pub?gid=212966958&single=true&output=csv")

# Task 3 — Death Percentage Ranking

### Q-3 — Country with the highest death rate

**Goal:** Compute, per country, `deaths / confirmed cases`, and report the country with the highest death percentage.

###`Q-3:` Show Country with Heighest death percent out of confirmed Cases. 

## 3.1 Approach — group and aggregate

The requested metric is a *ratio*: for each Country/Region, sum total deaths and divide by sum total confirmed cases, then multiply by 100.

Because the MultiIndex from Q-2 already has Country at level-0, the direct tool is a grouped aggregation — either construct the rates manually: take `datedata` (dropping the date dimension by summing over it per country) with `groupby('Country/Region')` on both the cases and deaths totals, then divide; or reset the index so Country becomes a plain column and use `groupby` for a clearer path.

A neat alternative that avoids double bookkeeping: reset the index, then `groupby('Country/Region')[['No of cases', 'No of Deaths']].sum()`, divide the two sums element-wise into a `death_percent` column, and finish with `.sort_values('death_percent', ascending=False)` to see the risk ranking at a glance.

Watch two details:
- **Zeros and missing values** — some provinces report no cases or are entirely missing; guard divisions with `fillna(0)` or leave them and note the `NaN`.
- **Interpretation** — countries with tiny case counts can show inflated percentages with a handful of deaths, so read the ranking critically rather than mechanically.

In [ ]:
# code here

# Task 4 — India's Daily New Cases

### Q-4 — A `No of new cases` column using `shift`

**Goal:** Make a DataFrame for India from the COVID data plus one extra column, `No of new cases`, defined as today's cases minus the previous day's cases. The first day gets `NaN` (or the value itself). The hint cell shows the `shift` trick.

### `Q-4` : Make a dataframe for India from Covid Data with one extra column representing no of new cases. 

* Just for Assumption "No of new cases" will be equal to difference of "no of cases" with previous day.
* First day new cases will be NaN or equal to no of cases

Say on 12/30/22 No of cases is - 44679608	and a day previous (12/29/22) no of cases is - 44679382.

Then for 12/30/22 -> No of New Cases  = 44679608 - 44679382 = 

Note:- Try using shift Function

Try using the shift function

```
s = pd.Series([1,2,3,4,5,6])
s #-> [1,2,3,4,5,6]
s.shift(1) #-> [NaN, 1,2,3,4,5]
s.shift(-1) #-> [2,3,4,5,6, NaN]
```

## 4.1 Approach — the shift technique

New cases is a *first difference* of the cumulative case count. Difference of a series `s` is literally `s - s.shift(1)`: shift moves every value one position down, and subtracting the shifted series from the original gives, for day `t`, `cases_t - cases_{t-1}`. Exactly the sample output — `s.shift(1)` → `[NaN, 1, 2, 3, 4, 5]` — so `s - s.shift(1)` → `[NaN, 1, 1, 1, 1, 1]`.

Steps for your answer:
1. Filter `datedata` (or the interim merged frame) for `Country == 'India'` — remember the index build in Q-2, so select with `.loc['India']` if the index survived, or filter a reset frame.
2. Sort chronologically by `date`, because `shift` only makes sense on an ordered series.
3. Compute `india['No of new cases'] = india['No of cases'] - india['No of cases'].shift(1)` — the `shift(1)` already introduces `NaN` on the first row, matching the note (first day new cases is `NaN` or equal to the case count).
4. Optionally `shift(-1)` if the convention must be reversed; the hint cell shows both directions.

Why does this matter in production? `shift`-based differences are how you compute daily new infections, daily revenue change, and most other 'delta' columns — and `diff()` is the convenience alias Pandas gives you for exactly this operation at scale.

In [ ]:
# code here

# Task 5 — World's Most Polluted Cities

### Q-5 — MultiIndex on Country and City/Town

**Goal:** Read the polluted-cities dataset from the tinyurl link, set a MultiIndex on `Country` and `City/Town`, and answer:
1. most polluted Indian city by **PM10**; 2. least polluted Indian city by **PM10**;
3. same two for China; 4. a pie chart of `PM10` for Poland;
5. a bar chart of `PM2.5` and `PM10` for Israel and Qatar; 6. collapse the frame to a Series without losing information.

### `Q-5:` Read the Dataset using the below given link and create a multi-index dataframe using the columns "Country" and "City/Town". 

This dataset is about the most polluted cities in the world. You can get details from [here](https://www.kaggle.com/datasets/rajkumarpandey02/worlds-most-air-polluted-countries-cities).

**Dataset link:** https://tinyurl.com/2fe6vz4u Directly use this link to read.

**Task:**
1. Find out the name of the city of India which is most poluted based on PM10.
2. Find out the name of the city of India which has minumum pollution level based on PM10.
3. Do same operations (like 1 and 2) with the country China.
4. Make a pie chart based on the column "PM10" of the country Poland.
5. Make a bar chart based on the columns "PM2.5" and "PM10" of the countries Israel and Qatar.
6. Convert this MultiIndex DataFrame to Series by retaining the informations.

## 5.1 Approach — loc lookups, group charts, and squeezing to a Series

The frame arrives with `Country` and `City/Town` columns, plus pollutant columns `PM2.5`, `PM10`, `NO2`, ..., and a dozen rows per country. Set the index with `set_index(['Country', 'City/Town'])` to unlock the whole task set.

**1 & 2.** For India, most and least polluted by PM10: select the country slice (`df.loc['India']`), then `sort_values('PM10', ascending=False)` to pick the first row (worst) and last (best). Equivalently `idxmax()`/`idxmin()` with `.loc` on the result.

**3.** Repeat the identical sort-and-slice for `df.loc['China']`.

**4.** Poland pie chart: pick `df.loc['Poland']`, extract its `PM10` Series, and plot with `kind='pie'`. A typical visualisation library renders percentage slices, letting you *see* which city dominates Poland's particulate burden. Check you have non-negative, non-NaN values before plotting (`fillna(0)` often needed).

**5.** Israel and Qatar bars: `df.loc[['Israel','Qatar']]` gives the countries' rows; select the two columns `['PM2.5', 'PM10']` and plot `kind='bar'` — one grouped bar cluster per row, so you can compare the two pollutants across both countries in a single chart.

**6.** Collapsing to a Series: the MultiIndex Series is natively stored as a long Series with `(Country, City)` stacked keys, but a DataFrame with one column (`df['PM10']`) already *is* a Series *with* the MultiIndex — the 'collapse without losing information' usually means one of `stack()` (if you swap axes) or simply selecting the single meaningful column. Aim to justify why no data is lost in your chosen form.

Throughout, watch the spellings (`City/Town` has a slash) and use `reset_index()`/`set_index()` freely as you explore interactively.

## Summary

This task set cemented the MultiIndex in five practical scenarios. Q-1 rebuilt a two-level DataFrame by swapping index/column levels (`swaplevel(axis=1)`), sorting, and transposing — pure structural surgery. Q-2 melted two wide COVID tables, merged them on location+date keys, and `set_index` a `Country → Province/State` hierarchy. Q-3 aggregated per-country deaths and cases to rank death percentages with `groupby` + `sort_values`. Q-4 leaned on the killer `shift` function to derive a *new cases* column as `cases - cases.shift(1)`, the standard first-difference pattern behind daily-change analytics. Q-5 spanned selection (`df.loc['India']`), sorting, and ggplot-style charts — pie for Poland's PM10, grouped bars for Israel/Qatar — and finished by collapsing to a tidy Series. Together they show the MultiIndex is not just a display flag: it structures real-world data so that one-liners such as `loc['India']` and per-country group charts become one-liners.

In [ ]:
# code here